# MarketSense · Exploratory analysis
Run from the project root or notebooks directory after acquiring the documented real dataset. No financial outputs are bundled. Design models from training-period EDA; reserve the test period for final evaluation.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import seaborn as sns
from config.settings import Settings
from src.data_loader import load_csv
from src.feature_engineering import engineer_features, create_labels, FEATURE_COLUMNS
from src.market_prediction import chronological_split
DATA = ROOT / 'data' / 'raw' / 'aapl.us.txt'  # change to your real single-stock file
if not DATA.exists():
    raise FileNotFoundError('Acquire the real stock CSV first; see docs/dataset.md')
bars, audit = load_csv(DATA)
audit

In [ ]:
features = engineer_features(bars)
split = chronological_split(create_labels(features), Settings())
training = split.train
print('Historical coverage:', bars.Date.min(), 'to', bars.Date.max())
print('Training EDA:', training.Date.min(), 'to', training.Date.max())
training[FEATURE_COLUMNS].describe().T

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, column in zip(axes, ['Close', 'daily_return', 'Volume']):
    ax.plot(training.Date, training[column])
    ax.set_ylabel(column)
fig.suptitle('Training-period observations · source price units, decimal returns')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.histplot(training.daily_return, bins=40, ax=axes[0])
axes[0].set_title('Training daily returns')
sns.heatmap(training[FEATURE_COLUMNS].corr(), cmap='vlag', center=0, ax=axes[1])
axes[1].set_title('Training feature correlations')
fig.tight_layout()
plt.show()
training.target.value_counts(normalize=True).rename(index={0.0: 'Bearish', 1.0: 'Bullish'})

## Questions for the report
- What rows were removed and why? Are there unexpected date gaps or zero volumes?
- Do extreme returns reflect market events, adjustments, or data problems? Verify against source records.
- Which indicators are redundant or highly correlated?
- How imbalanced are the direction labels in training?
- Keep modelling choices fixed before opening test scores. Generate genuine metrics through train.py rather than duplicating the pipeline here.

Optional notebook UI: install JupyterLab separately with `python -m pip install jupyterlab` and run `jupyter lab`.